# Generate teacher-requested DWI QC products

This notebook creates only QC derivatives that are not already guaranteed by the preprocessing scripts. It does not choose a better pipeline. Run one subject at a time, beginning with D0007.

Already produced by v2: noise map, denoising residual, corrected mean b0, brain mask, FA, MD, AD, and RD. The cells below skip existing products unless `OVERWRITE=True`.

## 1. Configuration

Confirm the v1 directory. The v2 path matches the revised script shared on 2026-09-02.

In [127]:
from pathlib import Path
import csv, json, shutil, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SUBJECT = 'D0007'
V1_DIR = (
    Path("/mnt/newStor/paros/paros_WORK/xintong/test/preprocess_v1_qc")
    / SUBJECT
)
V2_DIR = Path('/mnt/newStor/paros/paros_WORK/xintong/test/preprocess_v2') / SUBJECT
QC_ROOT = Path('/mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc') / SUBJECT
OVERWRITE = False

QC_V1 = QC_ROOT / 'v1'
QC_V2 = QC_ROOT / 'v2'
QC_COMPARE = QC_ROOT / 'v1_vs_v2'
FREEVIEW_DIR = QC_ROOT / 'freeview_images'
for directory in (QC_V1, QC_V2, QC_COMPARE, FREEVIEW_DIR):
    directory.mkdir(parents=True, exist_ok=True)

V1 = {
    "raw": V1_DIR / f"{SUBJECT}_01_dwi_raw.mif",
    "denoised": V1_DIR / f"{SUBJECT}_02_dwi_denoised.mif",
    "noise": (
        V1_DIR
        / "qc"
        / f"{SUBJECT}_noise.mif"
    ),
    "residual": (
        V1_DIR
        / "qc"
        / f"{SUBJECT}_denoising_residual.mif"
    ),
    "degibbs": (
        V1_DIR
        / f"{SUBJECT}_03_dwi_degibbs.mif"
    ),
    "preprocessed": (
        V1_DIR
        / f"{SUBJECT}_04_dwi_preprocessed.mif"
    ),
    "corrected_b0": (
        V1_DIR
        / f"{SUBJECT}_corrected_mean_b0.nii.gz"
    ),
    "eddy_mask": (
        V1_DIR
        / f"{SUBJECT}_eddy_mask.nii.gz"
    ),
    "biascorrected": (
        V1_DIR
        / f"{SUBJECT}_05_dwi_biascorrected.mif"
    ),
    "tensor": (
        V1_DIR
        / f"{SUBJECT}_tensor.mif"
    ),
    "fa": V1_DIR / f"{SUBJECT}_fa.nii.gz",
    "md": V1_DIR / f"{SUBJECT}_md.nii.gz",
    "ad": V1_DIR / f"{SUBJECT}_ad.nii.gz",
    "rd": V1_DIR / f"{SUBJECT}_rd.nii.gz",
    "eddy_scratch": V1_DIR / "eddy_scratch",
}
V2 = {
    'raw': V2_DIR / f'{SUBJECT}_01_raw.mif',
    'denoised': V2_DIR / f'{SUBJECT}_02_denoised.mif',
    'noise': V2_DIR / 'qc' / f'{SUBJECT}_noise.mif',
    'residual': V2_DIR / 'qc' / f'{SUBJECT}_denoising_residual.mif',
    'degibbs': V2_DIR / f'{SUBJECT}_03_degibbs.mif',
    'preprocessed': V2_DIR / f'{SUBJECT}_04_preprocessed.mif',
    'corrected_b0': V2_DIR / f'{SUBJECT}_corrected_mean_b0.nii.gz',
    'mask': V2_DIR / f'{SUBJECT}_corrected_mean_b0_brain_mask.nii.gz',
    'biascorrected': V2_DIR / f'{SUBJECT}_05_biascorrected.mif',
    'fa': V2_DIR / f'{SUBJECT}_fa.nii.gz',
    'md': V2_DIR / f'{SUBJECT}_md.nii.gz',
    'ad': V2_DIR / f'{SUBJECT}_ad.nii.gz',
    'rd': V2_DIR / f'{SUBJECT}_rd.nii.gz',
    'eddy_scratch': V2_DIR / 'eddy_scratch',
}
print('QC output:', QC_ROOT)

QC output: /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007


In [128]:
def run_product(command, output):
    output = Path(output)
    if output.exists() and not OVERWRITE:
        print('Exists, skipped:', output)
        return True
    missing_tools = [command[0]] if shutil.which(command[0]) is None else []
    if missing_tools:
        print('Missing command:', missing_tools[0])
        return False
    print('Running:', ' '.join(map(str, command)))
    subprocess.run([str(x) for x in command], check=True)
    return output.exists()

def require(*paths):
    missing = [str(p) for p in paths if not Path(p).exists()]
    if missing:
        raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))

## 2. Cohort-level Eddy validation

Checks acquisition/Eddy metadata, motion, outliers, and runs `eddy_quad` for all listed subjects and both pipelines. Missing inputs are skipped and recorded.

In [216]:
RUN_EDDY_QUAD = True
EDDY_QC_ROOT = Path('/mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/cohort/eddy_quad_reports')
EDDY_QC_ROOT.mkdir(parents=True, exist_ok=True)
def complete_eddy_dir(root):
    required = ['dwi_post_eddy.nii.gz', 'eddy_indices.txt', 'eddy_config.txt', 'eddy_mask.nii', 'bvals']
    candidates = sorted(Path(root).glob('dwifslpreproc-tmp-*')) if Path(root).exists() else []
    good = [d for d in candidates if all((d / f).exists() for f in required)]
    return good[-1] if good else None
def load_numeric(path, skiprows=0):
    if path is None or not path.exists() or path.stat().st_size == 0: return None
    try: return np.loadtxt(path, skiprows=skiprows, ndmin=2)
    except (OSError, ValueError): return None
eddy_rows = []
for subject in COHORT_SUBJECTS:
    for pipeline, root in [('v1', V1_ROOT / subject / 'eddy_scratch'), ('v2', V2_ROOT / subject / 'eddy_scratch')]:
        d = complete_eddy_dir(root)
        row = {'subject': subject, 'pipeline': pipeline, 'status': 'missing', 'scratch_dir': str(d) if d else ''}
        if d is not None:
            row['status'] = 'available'
            row['phase_encoding_config'] = (d / 'eddy_config.txt').read_text().strip()
            for label, filename in [('motion_rms', 'dwi_post_eddy.eddy_movement_rms'), ('restricted_rms', 'dwi_post_eddy.eddy_restricted_movement_rms')]:
                values = load_numeric(d / filename)
                if values is not None: row[f'{label}_mean'] = float(np.nanmean(values)); row[f'{label}_max'] = float(np.nanmax(values))
            values = load_numeric(d / 'dwi_post_eddy.eddy_outlier_map', skiprows=1)
            if values is not None: row['outlier_flagged'] = int(np.count_nonzero(values)); row['outlier_evaluated'] = int(values.size); row['outlier_percent'] = float(100 * np.mean(values != 0))
            if RUN_EDDY_QUAD and shutil.which('eddy_quad'):
                report = EDDY_QC_ROOT / subject / pipeline; report.mkdir(parents=True, exist_ok=True)
                command = ['eddy_quad', str(d / 'dwi_post_eddy'), '-idx', str(d / 'eddy_indices.txt'), '-par', str(d / 'eddy_config.txt'), '-m', str(d / 'eddy_mask.nii'), '-b', str(d / 'bvals'), '-o', str(report)]
                result = subprocess.run(command, capture_output=True, text=True)
                row['eddy_quad_status'] = 'completed' if result.returncode == 0 else f'failed: {result.stderr[-300:]}'
            elif RUN_EDDY_QUAD: row['eddy_quad_status'] = 'skipped: eddy_quad not found'
        eddy_rows.append(row)
eddy_qc = pd.DataFrame(eddy_rows)
cohort_eddy_summary = eddy_qc  # shared name for the downstream selection cell
eddy_qc.to_csv(COHORT_QC_ROOT / 'unified_eddy_validation_10_subjects.csv', index=False)
display(eddy_qc)

,subject,pipeline,status,scratch_dir,phase_encoding_config,motion_rms_mean,motion_rms_max,restricted_rms_mean,restricted_rms_max,outlier_flagged,outlier_evaluated,outlier_percent,eddy_quad_status
0,D0007,v1,available,/mnt/newStor/paros/paros_WORK/xintong/test/pre...,0 1 0 0.0762,0.351981,1.021767,0.332899,1.004140,0,3294,0.000000,"failed: s/eddy_qc/QUAD/quad.py"", line 162, in ..."
1,D0007,v2,available,/mnt/newStor/paros/paros_WORK/xintong/test/pre...,0 1 0 0.0762,0.344238,1.006012,0.325396,0.978002,0,3294,0.000000,"failed: s/eddy_qc/QUAD/quad.py"", line 162, in ..."
2,D0051,v1,available,/mnt/newStor/paros/paros_WORK/xintong/test/pre...,0 1 0 0.0762,0.172886,0.461851,0.158315,0.460793,0,3294,0.000000,"failed: s/eddy_qc/QUAD/quad.py"", line 162, in ..."
3,D0051,v2,available,/mnt/newStor/paros/paros_WORK/xintong/test/pre...,0 1 0 0.0762,0.171287,0.468295,0.156313,0.467527,0,3294,0.000000,"failed: s/eddy_qc/QUAD/quad.py"", line 162, in ..."
4,D0057,v1,available,/mnt/newStor/paros/paros_WORK/xintong/test/pre...,0 1 0 0.119,0.359175,1.070794,0.329304,1.058345,0,3294,0.000000,"failed: s/eddy_qc/QUAD/quad.py"", line 162, in ..."
5,D0057,v2,available,/mnt/newStor/paros/paros_WORK/xintong/test/pre...,0 1 0 0.119,0.363792,1.095921,0.333680,1.082191,0,3294,0.000000,"failed: s/eddy_qc/QUAD/quad.py"", line 162, in ..."
6,D0066,v1,available,/mnt/newStor/paros/paros_WORK/xintong/test/pre...,0 1 0 0.0762,0.146452,0.369877,0.135674,0.352151,0,3294,0.000000,"failed: s/eddy_qc/QUAD/quad.py"", line 162, in ..."
7,D0066,v2,available,/mnt/newStor/paros/paros_WORK/xintong/test/pre...,0 1 0 0.0762,0.142046,0.357252,0.130958,0.337835,0,3294,0.000000,"failed: s/eddy_qc/QUAD/quad.py"", line 162, in ..."
8,D0089,v1,available,/mnt/newStor/paros/paros_WORK/xintong/test/pre...,0 1 0 0.0762,0.258777,0.628662,0.235503,0.615956,0,3294,0.000000,"failed: s/eddy_qc/QUAD/quad.py"", line 162, in ..."
9,D0089,v2,available,/mnt/newStor/paros/paros_WORK/xintong/test/pre...,0 1 0 0.0762,0.257712,0.622906,0.234547,0.610635,0,3294,0.000000,"failed: s/eddy_qc/QUAD/quad.py"", line 162, in ..."


## 3. Denoising residuals

Teacher-requested product: `raw - denoised`. v2 already creates it; this cell creates the missing v1 residual. A noise map cannot be reconstructed by subtraction; v1 must rerun `dwidenoise -noise` if a v1 noise map is desired.

In [129]:
v1_residual = QC_V1 / f'{SUBJECT}_denoising_residual.mif'
require(V1['raw'], V1['denoised'])
run_product(['mrcalc', V1['raw'], V1['denoised'], '-subtract', v1_residual, '-force'], v1_residual)
print('v2 residual:', V2['residual'], 'exists =', V2['residual'].exists())

Exists, skipped: /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/v1/D0007_denoising_residual.mif
v2 residual: /mnt/newStor/paros/paros_WORK/xintong/test/preprocess_v2/D0007/qc/D0007_denoising_residual.mif exists = True


## 4. Matching 3D images for Gibbs inspection

Teacher-requested product: matched denoised and degibbs volumes for side-by-side inspection. Set `DWI_VOLUME` to the same diffusion volume for both pipelines.

In [130]:
DWI_VOLUME = 2
for label, source in [('v1_denoised', V1['denoised']), ('v1_degibbs', V1['degibbs']), ('v2_denoised', V2['denoised']), ('v2_degibbs', V2['degibbs'])]:
    destination = FREEVIEW_DIR / f'{SUBJECT}_{label}_vol{DWI_VOLUME}.nii.gz'
    require(source)
    run_product(['mrconvert', source, destination, '-coord', '3', str(DWI_VOLUME), '-force'], destination)

# Within-pipeline difference maps: each step minus the next step.
difference_pairs = [
    ('v1_denoising_residual', FREEVIEW_DIR / f'{SUBJECT}_v1_raw_vol{DWI_VOLUME}.nii.gz', FREEVIEW_DIR / f'{SUBJECT}_v1_denoised_vol{DWI_VOLUME}.nii.gz'),
    ('v1_gibbs_residual', FREEVIEW_DIR / f'{SUBJECT}_v1_denoised_vol{DWI_VOLUME}.nii.gz', FREEVIEW_DIR / f'{SUBJECT}_v1_degibbs_vol{DWI_VOLUME}.nii.gz'),
    ('v2_denoising_residual', FREEVIEW_DIR / f'{SUBJECT}_v2_raw_vol{DWI_VOLUME}.nii.gz', FREEVIEW_DIR / f'{SUBJECT}_v2_denoised_vol{DWI_VOLUME}.nii.gz'),
    ('v2_gibbs_residual', FREEVIEW_DIR / f'{SUBJECT}_v2_denoised_vol{DWI_VOLUME}.nii.gz', FREEVIEW_DIR / f'{SUBJECT}_v2_degibbs_vol{DWI_VOLUME}.nii.gz'),
]
for name, first_image, second_image in difference_pairs:
    difference = FREEVIEW_DIR / f'{SUBJECT}_{name}_vol{DWI_VOLUME}.nii.gz'
    require(first_image, second_image)
    run_product(['mrcalc', first_image, second_image, '-subtract', difference, '-force'], difference)
    print('Processing difference map:', difference)

Exists, skipped: /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/freeview_images/D0007_v1_denoised_vol2.nii.gz
Exists, skipped: /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/freeview_images/D0007_v1_degibbs_vol2.nii.gz
Exists, skipped: /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/freeview_images/D0007_v2_denoised_vol2.nii.gz
Exists, skipped: /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/freeview_images/D0007_v2_degibbs_vol2.nii.gz
Exists, skipped: /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/freeview_images/D0007_v1_denoising_residual_vol2.nii.gz
Processing difference map: /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/freeview_images/D0007_v1_denoising_residual_vol2.nii.gz
Exists, skipped: /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/freeview_images/D0007_v1_gibbs_residual_vol2.nii.gz
Processing difference map: /mnt/newStor/paros/paros_WORK/xinton

## 5. Corrected mean b0 for v1

Teacher-requested comparison product: v1 corrected mean b0. v2 already saves the equivalent image.

In [131]:
v1_corrected_b0 = QC_V1 / f'{SUBJECT}_corrected_mean_b0.nii.gz'
require(V1['preprocessed'])
if not v1_corrected_b0.exists() or OVERWRITE:
    command = f"dwiextract '{V1['preprocessed']}' -bzero - | mrmath - mean '{v1_corrected_b0}' -axis 3 -force"
    subprocess.run(command, shell=True, executable='/bin/bash', check=True)
print('v1 corrected b0:', v1_corrected_b0)
print('v2 corrected b0:', V2['corrected_b0'])

v1 corrected b0: /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/v1/D0007_corrected_mean_b0.nii.gz
v2 corrected b0: /mnt/newStor/paros/paros_WORK/xintong/test/preprocess_v2/D0007/D0007_corrected_mean_b0.nii.gz


## 6. Mask comparison products

Teacher-requested product: mask overlap summary and a disagreement mask. This requires v1 and v2 masks to be on the same voxel grid. Dice measures agreement, not anatomical correctness; the disagreement image must still be inspected on mean b0.

In [136]:
V1_MASK = V1['eddy_mask']
V2_MASK = V2['mask']
mask_xor = QC_COMPARE / f'{SUBJECT}_mask_disagreement.nii.gz'
mask_stats = QC_COMPARE / f'{SUBJECT}_mask_overlap.csv'
require(V1_MASK, V2_MASK)
run_product(['mrcalc', V1_MASK, V2_MASK, '-xor', mask_xor, '-datatype', 'bit', '-force'], mask_xor)

def count_nonzero_voxels(path):
    result = subprocess.run(['mrstats', str(path), '-mask', str(path), '-output', 'count'], check=True, capture_output=True, text=True)
    return float(result.stdout.strip())

intersection = QC_COMPARE / f'{SUBJECT}_mask_intersection.nii.gz'
run_product(['mrcalc', V1_MASK, V2_MASK, '-mult', intersection, '-datatype', 'bit', '-force'], intersection)
n1, n2, ni = [count_nonzero_voxels(p) for p in (V1_MASK, V2_MASK, intersection)]
dice = 2 * ni / (n1 + n2) if (n1 + n2) else np.nan
pd.DataFrame([{'subject': SUBJECT, 'v1_voxels': n1, 'v2_voxels': n2, 'intersection_voxels': ni, 'dice': dice}]).to_csv(mask_stats, index=False)
display(pd.read_csv(mask_stats))

Exists, skipped: /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/v1_vs_v2/D0007_mask_disagreement.nii.gz
Exists, skipped: /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/v1_vs_v2/D0007_mask_intersection.nii.gz


,subject,v1_voxels,v2_voxels,intersection_voxels,dice
0,D0007,1470842.0,1350094.0,1307195.0,0.926781


## 7. FA/MD/AD/RD difference maps

Teacher-requested comparison products. Configure v1 metric paths below. Images must already share the same grid; this notebook stops rather than silently resampling quantitative maps.

In [137]:
V1_METRICS = {
    'fa': V1_DIR / f'{SUBJECT}_fa.nii.gz',
    'md': V1_DIR / f'{SUBJECT}_md.nii.gz',
    'ad': V1_DIR / f'{SUBJECT}_ad.nii.gz',
    'rd': V1_DIR / f'{SUBJECT}_rd.nii.gz',
}
# V1 historical output uses adc for mean diffusivity; check it first.
V1_METRICS['md'] = V1_DIR / f'{SUBJECT}_adc.nii.gz' if (V1_DIR / f'{SUBJECT}_adc.nii.gz').exists() else V1_DIR / f'{SUBJECT}_md.nii.gz'
rows = []
for metric in ('fa', 'md', 'ad', 'rd'):
    old, new = V1_METRICS[metric], V2[metric]
    if not old.exists() or not new.exists():
        rows.append({'metric': metric.upper(), 'status': 'missing input', 'v1': str(old), 'v2': str(new)})
        continue
    difference = QC_COMPARE / f'{SUBJECT}_{metric}_v2_minus_v1.nii.gz'
    run_product(['mrcalc', new, old, '-subtract', difference, '-force'], difference)
    rows.append({'metric': metric.upper(), 'status': 'created', 'difference_map': str(difference)})
metric_products = pd.DataFrame(rows)
metric_products.to_csv(QC_COMPARE / f'{SUBJECT}_metric_product_status.csv', index=False)
display(metric_products)

,metric,status,v1,v2
0,FA,missing input,/mnt/newStor/paros/paros_WORK/xintong/test/pre...,/mnt/newStor/paros/paros_WORK/xintong/test/pre...
1,MD,missing input,/mnt/newStor/paros/paros_WORK/xintong/test/pre...,/mnt/newStor/paros/paros_WORK/xintong/test/pre...
2,AD,missing input,/mnt/newStor/paros/paros_WORK/xintong/test/pre...,/mnt/newStor/paros/paros_WORK/xintong/test/pre...
3,RD,missing input,/mnt/newStor/paros/paros_WORK/xintong/test/pre...,/mnt/newStor/paros/paros_WORK/xintong/test/pre...


## 8. Regional diffusion stability

Teacher-requested product, but it requires one atlas/label image already transformed into the diffusion grid. Set `ATLAS_IN_DWI` only after registration is verified. The output contains v1, v2, absolute difference, and percent difference for every nonzero label.

In [138]:
ATLAS_IN_DWI = None  # Example: Path('/path/to/atlas_in_DWI_space.nii.gz')
if ATLAS_IN_DWI is None:
    print('Skipped: set ATLAS_IN_DWI after anatomical alignment is verified.')
else:
    atlas = Path(ATLAS_IN_DWI)
    require(atlas)
    regional_rows = []
    for metric in ('fa', 'md', 'ad', 'rd'):
        old, new = V1_METRICS[metric], V2[metric]
        require(old, new)
        labels_text = subprocess.run(['mrstats', str(atlas), '-output', 'max'], check=True, capture_output=True, text=True).stdout
        max_label = int(float(labels_text.strip()))
        for label in range(1, max_label + 1):
            temp_mask = QC_COMPARE / f'_tmp_label_{label}.mif'
            subprocess.run(['mrcalc', atlas, str(label), '-eq', temp_mask, '-force'], check=True)
            means = []
            for image in (old, new):
                value = subprocess.run(['mrstats', str(image), '-mask', str(temp_mask), '-output', 'mean'], check=True, capture_output=True, text=True)
                means.append(float(value.stdout.strip()))
            if np.isfinite(means).all():
                diff = means[1] - means[0]
                pct = 100 * diff / means[0] if means[0] != 0 else np.nan
                regional_rows.append({'subject': SUBJECT, 'metric': metric.upper(), 'label': label, 'v1_mean': means[0], 'v2_mean': means[1], 'difference': diff, 'percent_difference': pct})
            temp_mask.unlink(missing_ok=True)
    regional = pd.DataFrame(regional_rows)
    regional.to_csv(QC_COMPARE / f'{SUBJECT}_regional_diffusion_stability.csv', index=False)
    display(regional.head())

Skipped: set ATLAS_IN_DWI after anatomical alignment is verified.


## Products that still require a preserved/re-run eddy job

The notebook cannot reconstruct motion parameters, model-fit residuals, or a valid eddy_quad report from the final preprocessed image. If the required files are absent from the inventory, rerun that pipeline while preserving/exporting eddy outputs. Anatomical-alignment products also require the confirmed DWI-to-T1 transform. Tractography/connectome reproducibility is explicitly deferred (`if needed later`).

## Convert comparison images for Freeview

Run the next cell on a node with `mrconvert` available. All Freeview images are placed in one folder. This only converts/extracts images and does not rerun preprocessing.

In [ ]:
FREEVIEW_DIR = QC_ROOT / 'freeview_images'
FREEVIEW_DIR.mkdir(parents=True, exist_ok=True)
image_conversions = [
    (V1['raw'], FREEVIEW_DIR / f'{SUBJECT}_v1_raw_vol2.nii.gz', 2),
    (V2['raw'], FREEVIEW_DIR / f'{SUBJECT}_v2_raw_vol2.nii.gz', 2),
    (V1['denoised'], FREEVIEW_DIR / f'{SUBJECT}_v1_denoised_vol2.nii.gz', 2),
    (V1['degibbs'], FREEVIEW_DIR / f'{SUBJECT}_v1_degibbs_vol2.nii.gz', 2),
    (V2['denoised'], FREEVIEW_DIR / f'{SUBJECT}_v2_denoised_vol2.nii.gz', 2),
    (V2['degibbs'], FREEVIEW_DIR / f'{SUBJECT}_v2_degibbs_vol2.nii.gz', 2),
]
for source, destination, volume in image_conversions:
    require(source)
    run_product(['mrconvert', source, destination, '-coord', '3', str(volume), '-force'], destination)
    print('Freeview image:', destination)

residual_conversions = [(V1['residual'], FREEVIEW_DIR / f'{SUBJECT}_v1_denoising_residual.nii.gz'), (V2['residual'], FREEVIEW_DIR / f'{SUBJECT}_v2_denoising_residual.nii.gz')]
for source, destination in residual_conversions:
    require(source)
    run_product(['mrconvert', source, destination, '-force'], destination)
    print('Freeview file:', destination)

Running: mrconvert /mnt/newStor/paros/paros_WORK/xintong/test/preprocess_v1_qc/D0007/D0007_01_dwi_raw.mif /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/freeview_images/D0007_v1_raw_vol2.nii.gz -coord 3 2 -force


mrconvert: [WARNING] existing output files will be overwritten
mrconvert: [100%] copying from "/mnt/newSt.../D0007/D0007_01_dwi_raw.mif" to "/mnt/newSt...es/D0007_v1_raw_vol2.nii.gz"[0K
mrconvert: [100%] compressing image "/mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/freeview_images/D0007_v1_raw_vol2.nii.gz"[0K
mrconvert: [WARNING] existing output files will be overwritten
mrconvert: [100%] copying from "/mnt/newSt...s_v2/D0007/D0007_01_raw.mif" to "/mnt/newSt...es/D0007_v2_raw_vol2.nii.gz"[0K
mrconvert: [  2%] compressing image "/mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/freeview_images/D0007_v2_raw_vol2.nii.gz"...

Freeview image: /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/freeview_images/D0007_v1_raw_vol2.nii.gz
Running: mrconvert /mnt/newStor/paros/paros_WORK/xintong/test/preprocess_v2/D0007/D0007_01_raw.mif /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/freeview_images/D0007_v2_raw_vol2.nii.gz -coord 3 2 -force


mrconvert: [100%] compressing image "/mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/freeview_images/D0007_v2_raw_vol2.nii.gz"[0K
mrconvert: [WARNING] existing output files will be overwritten
mrconvert: [100%] copying from "/mnt/newSt...7/D0007_02_dwi_denoised.mif" to "/mnt/newSt...007_v1_denoised_vol2.nii.gz"[0K
mrconvert: [ 14%] compressing image "/mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/freeview_images/D0007_v1_denoised_vol2.nii.gz"...

Freeview image: /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/freeview_images/D0007_v2_raw_vol2.nii.gz
Running: mrconvert /mnt/newStor/paros/paros_WORK/xintong/test/preprocess_v1_qc/D0007/D0007_02_dwi_denoised.mif /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/freeview_images/D0007_v1_denoised_vol2.nii.gz -coord 3 2 -force


mrconvert: [100%] compressing image "/mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/freeview_images/D0007_v1_denoised_vol2.nii.gz"[0K
mrconvert: [WARNING] existing output files will be overwritten
mrconvert: [100%] copying from "/mnt/newSt...07/D0007_03_dwi_degibbs.mif" to "/mnt/newSt...0007_v1_degibbs_vol2.nii.gz"[0K
mrconvert: [ 10%] compressing image "/mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/freeview_images/D0007_v1_degibbs_vol2.nii.gz"...

... (8 outputs left out)

Freeview file: /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/freeview_images/D0007_v1_denoising_residual.nii.gz
Running: mrconvert /mnt/newStor/paros/paros_WORK/xintong/test/preprocess_v2/D0007/qc/D0007_denoising_residual.mif /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/freeview_images/D0007_v2_denoising_residual.nii.gz -force


mrconvert: [100%] compressing image "/mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/freeview_images/D0007_v2_denoising_residual.nii.gz"...

Freeview file: /mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/freeview_images/D0007_v2_denoising_residual.nii.gz


mrconvert: [100%] compressing image "/mnt/newStor/paros/paros_WORK/xintong/test/preprocessing_qc/D0007/freeview_images/D0007_v2_denoising_residual.nii.gz"


## Cohort-level preprocessing QC availability

This inventory checks the products needed for denoising, Gibbs, corrected-b0, mask, and metric QC. Existing products are included; missing products are recorded and skipped.

In [ ]:
def cohort_product_paths(subject, pipeline):
    root = (V1_ROOT if pipeline == 'v1' else V2_ROOT) / subject
    if pipeline == 'v1':
        return {'raw': root / f'{subject}_01_dwi_raw.mif', 'denoised': root / f'{subject}_02_dwi_denoised.mif', 'degibbs': root / f'{subject}_03_dwi_degibbs.mif', 'residual': root / 'qc' / f'{subject}_denoising_residual.mif', 'corrected_b0': root / f'{subject}_corrected_mean_b0.nii.gz', 'mask': root / f'{subject}_eddy_mask.nii.gz', 'fa': root / f'{subject}_fa.nii.gz', 'md': root / f'{subject}_adc.nii.gz' if (root / f'{subject}_adc.nii.gz').exists() else root / f'{subject}_md.nii.gz', 'ad': root / f'{subject}_ad.nii.gz', 'rd': root / f'{subject}_rd.nii.gz'}
    return {'raw': root / f'{subject}_01_raw.mif', 'denoised': root / f'{subject}_02_denoised.mif', 'degibbs': root / f'{subject}_03_degibbs.mif', 'residual': root / 'qc' / f'{subject}_denoising_residual.mif', 'corrected_b0': root / f'{subject}_corrected_mean_b0.nii.gz', 'mask': root / f'{subject}_corrected_mean_b0_brain_mask.nii.gz', 'fa': root / f'{subject}_fa.nii.gz', 'md': root / f'{subject}_md.nii.gz', 'ad': root / f'{subject}_ad.nii.gz', 'rd': root / f'{subject}_rd.nii.gz'}

availability_rows = []
for subject in COHORT_SUBJECTS:
    for pipeline in ('v1', 'v2'):
        products = cohort_product_paths(subject, pipeline)
        present = [name for name, path in products.items() if path.exists()]
        missing = [name for name, path in products.items() if not path.exists()]
        availability_rows.append({'subject': subject, 'pipeline': pipeline, 'status': 'complete' if not missing else ('partial' if present else 'missing'), 'present_products': ', '.join(present), 'missing_products': ', '.join(missing)})
availability = pd.DataFrame(availability_rows)
availability_csv = COHORT_QC_ROOT / 'preprocessing_qc_availability_10_subjects.csv'
availability.to_csv(availability_csv, index=False)
display(availability)
print('Saved:', availability_csv)

## Select representative subjects for visual presentation

Select one low-motion, one typical, and one high-motion subject from the available cohort. This is an objective Eddy-based shortlist; confirm visible artifacts in Freeview before finalizing.

In [ ]:
selection = cohort_eddy_summary.copy()
numeric_columns = ['movement_rms_mean', 'restricted_rms_mean', 'outlier_percent']
for column in numeric_columns:
    selection[column] = pd.to_numeric(selection.get(column), errors='coerce')
selection['available_metrics'] = selection[numeric_columns].notna().sum(axis=1)
selection = selection[selection['available_metrics'] > 0].copy()
subject_scores = selection.groupby('subject').agg(
    movement_rms=('movement_rms_mean', 'mean'),
    restricted_rms=('restricted_rms_mean', 'mean'),
    outlier_percent=('outlier_percent', 'mean'),
    available_pipelines=('pipeline', 'nunique'),
    available_metrics=('available_metrics', 'sum'),
).reset_index()
# Rank subjects using the available Eddy indicators; lower means less motion/outliers.
for column in ('movement_rms', 'restricted_rms', 'outlier_percent'):
    values = subject_scores[column]
    subject_scores[f'{column}_rank'] = values.rank(pct=True, na_option='keep')
rank_columns = [f'{column}_rank' for column in ('movement_rms', 'restricted_rms', 'outlier_percent')]
subject_scores['qc_motion_score'] = subject_scores[rank_columns].mean(axis=1)
subject_scores = subject_scores.sort_values('qc_motion_score')

low_motion = subject_scores.iloc[0] if len(subject_scores) else None
high_motion = subject_scores.iloc[-1] if len(subject_scores) else None
typical_index = len(subject_scores) // 2
typical = subject_scores.iloc[typical_index] if len(subject_scores) else None
recommendations = pd.DataFrame([
    {'role': 'low-motion / good-quality candidate', **low_motion.to_dict()} if low_motion is not None else {'role': 'low-motion / good-quality candidate'},
    {'role': 'typical / medium-quality candidate', **typical.to_dict()} if typical is not None else {'role': 'typical / medium-quality candidate'},
    {'role': 'high-motion / artifact-review candidate', **high_motion.to_dict()} if high_motion is not None else {'role': 'high-motion / artifact-review candidate'},
])
display(recommendations[['role', 'subject', 'qc_motion_score', 'movement_rms', 'restricted_rms', 'outlier_percent', 'available_pipelines']])
selection_csv = COHORT_QC_ROOT / 'representative_subject_selection.csv'
recommendations.to_csv(selection_csv, index=False)
print('Saved:', selection_csv)
print('Confirm the three candidates visually in Freeview before reporting them.')

## Unified cohort Eddy validation: metadata, motion, outliers, and eddy_quad

This is the definitive cohort Eddy section. It automatically selects a complete preserved Eddy scratch directory for each subject and pipeline, skips missing inputs, and records all statuses. Set `RUN_EDDY_QUAD=True` to generate the standard FSL reports.

In [ ]:
RUN_EDDY_QUAD = True
EDDY_QC_ROOT = COHORT_QC_ROOT / 'eddy_quad_reports'
EDDY_QC_ROOT.mkdir(parents=True, exist_ok=True)

def complete_eddy_dir(root):
    candidates = sorted(Path(root).glob('dwifslpreproc-tmp-*')) if Path(root).exists() else []
    required = ['dwi_post_eddy.nii.gz', 'eddy_indices.txt', 'eddy_config.txt', 'eddy_mask.nii', 'bvals']
    complete = [d for d in candidates if all((d / name).exists() for name in required)]
    return complete[-1] if complete else None

def numeric_file(path, skiprows=0):
    if path is None or not path.exists() or path.stat().st_size == 0:
        return None
    try:
        return np.loadtxt(path, skiprows=skiprows, ndmin=2)
    except (OSError, ValueError):
        return None

eddy_rows = []
for subject in COHORT_SUBJECTS:
    for pipeline, root in [('v1', V1_ROOT / subject / 'eddy_scratch'), ('v2', V2_ROOT / subject / 'eddy_scratch')]:
        scratch = complete_eddy_dir(root)
        row = {'subject': subject, 'pipeline': pipeline, 'status': 'missing', 'scratch_dir': str(scratch) if scratch else ''}
        if scratch is not None:
            row.update({'phase_encoding_config': (scratch / 'eddy_config.txt').read_text().strip(), 'index_file': str(scratch / 'eddy_indices.txt'), 'bvals_file': str(scratch / 'bvals')})
            motion = numeric_file(scratch / 'dwi_post_eddy.eddy_movement_rms')
            restricted = numeric_file(scratch / 'dwi_post_eddy.eddy_restricted_movement_rms')
            outlier = numeric_file(scratch / 'dwi_post_eddy.eddy_outlier_map', skiprows=1)
            row['status'] = 'available'
            if motion is not None:
                row.update({'movement_rms_mean': float(np.nanmean(motion)), 'movement_rms_max': float(np.nanmax(motion))})
            if restricted is not None:
                row.update({'restricted_rms_mean': float(np.nanmean(restricted)), 'restricted_rms_max': float(np.nanmax(restricted))})
            if outlier is not None:
                row.update({'outlier_flagged': int(np.count_nonzero(outlier)), 'outlier_evaluated': int(outlier.size), 'outlier_percent': float(100 * np.mean(outlier != 0))})
            if RUN_EDDY_QUAD and shutil.which('eddy_quad') is not None:
                report_dir = EDDY_QC_ROOT / subject / pipeline
                report_dir.mkdir(parents=True, exist_ok=True)
                command = ['eddy_quad', str(scratch / 'dwi_post_eddy'), '-idx', str(scratch / 'eddy_indices.txt'), '-par', str(scratch / 'eddy_config.txt'), '-m', str(scratch / 'eddy_mask.nii'), '-b', str(scratch / 'bvals'), '-o', str(report_dir)]
                result = subprocess.run(command, capture_output=True, text=True)
                row['eddy_quad_status'] = 'completed' if result.returncode == 0 else f'failed: {result.stderr[-300:]}'
            elif RUN_EDDY_QUAD:
                row['eddy_quad_status'] = 'skipped: eddy_quad not found'
            else:
                row['eddy_quad_status'] = 'not requested'
        eddy_rows.append(row)

unified_eddy = pd.DataFrame(eddy_rows)
unified_eddy_csv = COHORT_QC_ROOT / 'unified_eddy_validation_10_subjects.csv'
unified_eddy.to_csv(unified_eddy_csv, index=False)
display(unified_eddy)
print('Saved:', unified_eddy_csv)
print('Eddy_quad reports:', EDDY_QC_ROOT)